In [ ]:
from pathlib import Path
import sys

project_root = Path('c:/PdM').resolve()
sys.path.insert(0, str(project_root))

from src.data_simulation.physics.vibration_enhanced import EnhancedVibrationGenerator, BearingGeometry

gen = EnhancedVibrationGenerator()
signal, metrics = gen.generate_bearing_vibration(rpm=3000, bearing_health=0.80)

print(f"RMS: {metrics['rms']:.2f} mm/s")
print(f"Peak: {metrics['peak']:.2f} mm/s")
print(f"Crest Factor: {metrics['crest_factor']:.2f}")
print(f"Kurtosis: {metrics['kurtosis']:.2f}")

In [ ]:
# Large turbine bearing
large_bearing = BearingGeometry(
    n_balls=16,
    ball_diameter=25.0,
    pitch_diameter=150.0,
    contact_angle=0.0
)

gen = EnhancedVibrationGenerator(
    sample_rate=10240,
    resonance_freq=2500,
    bearing_geometry=large_bearing
)

signal, metrics = gen.generate_bearing_vibration(rpm=5000, bearing_health=0.65)

print(f"RMS: {metrics['rms']:.2f} mm/s")
print(f"Peak: {metrics['peak']:.2f} mm/s")
print(f"Crest Factor: {metrics['crest_factor']:.2f}")
print(f"Kurtosis: {metrics['kurtosis']:.2f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gen = EnhancedVibrationGenerator()

# Simulate progressive degradation
health_values = np.linspace(1.0, 0.3, 50)
rms_values = []
crest_values = []
kurt_values = []

for health in health_values:
    signal, metrics = gen.generate_bearing_vibration(
        rpm=3000,
        bearing_health=health,
        duration=1.0
    )
    rms_values.append(metrics['rms'])
    crest_values.append(metrics['crest_factor'])
    kurt_values.append(metrics['kurtosis'])

# Plot degradation trends
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

axes[0].plot(health_values, rms_values)
axes[0].set_ylabel('RMS (mm/s)')
axes[0].set_title('Vibration Trends vs Bearing Health')

axes[1].plot(health_values, crest_values)
axes[1].set_ylabel('Crest Factor')

axes[2].plot(health_values, kurt_values)
axes[2].set_ylabel('Kurtosis')
axes[2].set_xlabel('Bearing Health (1.0 = healthy)')

plt.tight_layout()
plt.show()

In [2]:
from pathlib import Path
import os
from datetime import datetime
import sys
project_root = Path('c:/PdM').resolve()
sys.path.insert(0, str(project_root))
from src.data_simulation.physics.environmental_conditions import EnvironmentalConditions, LocationType
from src.data_simulation.physics.weather_api_client import create_hybrid_environment

# Example 1: Synthetic fallback (no API key needed)
print("\n SYNTHETIC ENVIRONMENT")
synthetic = EnvironmentalConditions(location_type=LocationType.TROPICAL)
conditions = synthetic.get_conditions(elapsed_hours=326)
print(f"Tropical location (synthetic):")
print(f"  Temp: {conditions['ambient_temp_C']:.1f}°C")
print(f"  Humidity: {conditions['humidity_percent']:.1f}%")
print(f"  Dust exposure factor: {conditions['fouling_factor']:.2f}x")

# Example 2: Hybrid with real API (requires API key)
print("\nHYBRID ENVIRONMENT")

api_key = os.environ.get('WEATHER_API_KEY', '')

if api_key:
    print("\nAPI key found - testing real weather fetch...")

    # Example with location name (more user-friendly)
    hybrid_env = create_hybrid_environment(
        use_real_weather=True,
        api_provider="weatherapi",
        api_key=api_key,
        location_name="Bonny island",
        country="Nigeria",
        cache_enabled=True
    )

    try:
        real_conditions = hybrid_env.get_conditions(0, datetime.now())
        # print(real_conditions)
        print(f"\nBonny Nigeria (real weather):")
        print(f"  Temp: {real_conditions['ambient_temp_C']:.1f}°C")
        print(f"  Humidity: {real_conditions['humidity_percent']:.1f}%")
        print(f"  Pressure: {real_conditions['pressure_kPa']:.1f} kPa")
        print(f"  Wind Speed: {real_conditions['wind_speed_m_s']:.1f} m/s")
        print(f"  Dust exposure factor: {real_conditions['fouling_factor']:.2f}x")

        # Test cache
        cached_conditions = hybrid_env.get_conditions(0, datetime.now())
        print(f"\n  (Second call retrieved from cache)")
        print(cached_conditions)

    except Exception as e:
        print(f"\nAPI call failed: {e}")
        print("Falling back to synthetic data would occur automatically")
else:
    print("\n(No API key configured - skipping real weather test)")
    print("\nExample usage with location name:")
    print("  hybrid_env = create_hybrid_environment(")
    print("      use_real_weather=True,")
    print("      api_provider='weatherapi',")
    print("      api_key='your_key',")
    print("      location_name='Port Harcourt',")
    print("      country='Nigeria'")
    print("  )")

# Example 3: Cache preloading for offline simulation
print("\CACHE PRELOADING")
print("For reproducible offline simulations:")
print("  1. Preload cache with historical weather")
print("  2. Run simulation offline using cached data")
print("  3. Cache persists across runs for reproducibility")


 SYNTHETIC ENVIRONMENT
Tropical location (synthetic):
  Temp: 27.3°C
  Humidity: 78.8%
  Dust exposure factor: 1.24x

HYBRID ENVIRONMENT

API key found - testing real weather fetch...

Bonny Nigeria (real weather):
  Temp: 26.7°C
  Humidity: 72.0%
  Pressure: 100.8 kPa

API call failed: 'wind_speed_m_s'
Falling back to synthetic data would occur automatically
\CACHE PRELOADING
For reproducible offline simulations:
  1. Preload cache with historical weather
  2. Run simulation offline using cached data
  3. Cache persists across runs for reproducibility
